# Stipple G4 smoke test — polygon lasso

1M points, 3 clusters. Shift+drag to lasso a region; selected row indices arrive in `w.selected_indices`.

Gate target: round-trip < 200 ms on 1M-point select.

In [1]:
import numpy as np
from stipple import Stipple
import stipple
print(f'stipple {stipple.__version__}')

stipple 0.1.0.dev0


In [2]:
rng = np.random.default_rng(123)
n = 1_000_000
n_per = n // 3
labels = np.repeat(['a', 'b', 'c'], n_per)
if labels.shape[0] < n:
    labels = np.concatenate([labels, np.full(n - labels.shape[0], 'a')])
centers = np.array([[-3.0, 0.0], [3.0, 0.0], [0.0, 3.0]], dtype=np.float32)
x = np.empty(n, dtype=np.float32)
y = np.empty(n, dtype=np.float32)
for k in range(3):
    sl = slice(k * n_per, (k + 1) * n_per)
    x[sl] = centers[k, 0] + rng.standard_normal(n_per).astype(np.float32) * 0.5
    y[sl] = centers[k, 1] + rng.standard_normal(n_per).astype(np.float32) * 0.5
if n_per * 3 < n:
    x[n_per*3:] = centers[0, 0]
    y[n_per*3:] = centers[0, 1]
print(f'{n:,} points · 3 clusters · cluster_a indices = [0, {n_per:,})')

1,000,000 points · 3 clusters · cluster_a indices = [0, 333,333)


In [3]:
def on_selection(change):
    n_sel = len(w.selected_indices)
    if n_sel == 0:
        return
    in_cluster_a = (w.selected_indices < n_per).sum()
    purity = in_cluster_a / max(1, n_sel)
    print(f'selection #{w.selection_count}: {n_sel:,} rows · {purity:.1%} from cluster_a · {w.selection_ms:.1f} ms')

w = Stipple(x=x, y=y, color=labels)
w.observe(on_selection, names='selection_count')
w

In [4]:
import time
for _ in range(40):
    if w.last_fps:
        break
    time.sleep(0.1)
print(f'status: {w.status}')
print(f'FPS   : {w.last_fps:.1f}')
print('shift+drag in the canvas to lasso')

status: checking
FPS   : 0.0
shift+drag in the canvas to lasso
